# Latent Dirichlet Allocation

## Setup and Imports

In [169]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation as LDA, NMF 

In [170]:
sns.set_theme(style="white")
colors = "YlGnBu"

In [171]:
model_type = 'lda' # or 'nmf'
data_home = "../input"
output_dir = "../working"

In [172]:
OHCO = ['doc_title', 'para_num', 'sentence_num', 'token_num']
SENTS = OHCO[:3]
PARAS = OHCO[:2]
STORIES = OHCO[:1]

BAG = PARAS

In [173]:
BAG

['doc_title', 'para_num']

In [174]:
TOKENS = pd.read_csv('data/p2591-TOKENS.csv').set_index(OHCO).dropna()
TOKENS

pos_tuple  pos token_str  \
doc_title para_num sentence_num token_num                                   
ASHPUTTEL 0        0            0            ('The', 'DT')   DT       The   
                                1           ('wife', 'NN')   NN      wife   
                                2             ('of', 'IN')   IN        of   
                                3              ('a', 'DT')   DT         a   
                                4           ('rich', 'JJ')   JJ      rich   
...                                                    ...  ...       ...   
TOM THUMB 21       3            40            ('s', 'VBZ')  VBZ         s   
                                41            ('no', 'DT')   DT        no   
                                42         ('place', 'NN')   NN     place   
                                43          ('like', 'IN')   IN      like   
                                44          ('HOME', 'NN')   NN      HOME   

                                          term_str pos_group  
doc_title para_num sentence_num token_num                     
ASHPUTTEL 0        0            0              the        DT  
                                1             wife        NN  
                                2               of        IN  
                                3                a        DT  
                                4             rich        JJ  
...                                            ...       ...  
TOM THUMB 21       3            40               s        VB  
                                41              no        DT  
                                42           place        NN  
                                43            like        IN  
                                44            home        NN  

[101046 rows x 5 columns]

In [175]:
DOCS = TOKENS[TOKENS.pos.str.match(r'^NNS?$')]\
    .groupby(BAG).term_str\
    .apply(lambda x: ' '.join(map(str,x)))\
    .to_frame()\
    .rename(columns={'term_str':'doc_str'})

DOCS

doc_str
doc_title para_num                                                   
ASHPUTTEL 0         wife man end drew daughter girl watch afterwar...
          1         work daylight water fire sisters sorts ways ev...
          2         father fair wife s daughters clothes diamonds ...
          3         king land feast days son bride sisters hair sh...
          4                             peas ashes maiden door garden
...                                                               ...
TOM THUMB 17        wolf night house drain kitchen pantry ate dran...
          18        shout noise wolf everybody house clatter man mind
          19        woodman wife noise crack door wolf woodman axe...
          20                                             riches world
          21        son plenty clothes ones journey home father mo...

[889 rows x 1 columns]

## Create Vector Space

In [176]:
from sklearn.feature_extraction import text

my_stop_words = list(text.ENGLISH_STOP_WORDS.union(['yes']))
my_stop_words[:10]

['thin',
 'were',
 'couldnt',
 'can',
 'becoming',
 'etc',
 'for',
 'an',
 'while',
 'our']

In [177]:
count_engine = CountVectorizer(max_df=.9, min_df=10, stop_words=my_stop_words)
count_model = count_engine.fit_transform(DOCS.doc_str)
TERMS = count_engine.get_feature_names_out()
VOCAB = pd.DataFrame(index=TERMS)
VOCAB.index.name = 'term_str'
DTM = pd.DataFrame(count_model.toarray(), index=DOCS.index, columns=TERMS)
DTM

advice  air  alas  apple  arms  art  ashes  ass  ate  \
doc_title para_num                                                         
ASHPUTTEL 0              0    0     0      0     0    0      0    0    0   
          1              0    0     0      0     0    0      1    0    0   
          2              0    0     0      0     0    0      0    0    0   
          3              0    0     0      0     0    0      0    0    0   
          4              0    0     0      0     0    0      1    0    0   
...                    ...  ...   ...    ...   ...  ...    ...  ...  ...   
TOM THUMB 17             0    0     0      0     0    0      0    0    1   
          18             0    0     0      0     0    0      0    0    0   
          19             0    1     0      0     0    0      0    0    0   
          20             0    0     0      0     0    0      0    0    0   
          21             0    0     0      0     0    0      0    0    0   

                    bargain  ...  wolf  woman  wood  word  words  work  world  \
doc_title para_num           ...                                                
ASHPUTTEL 0               0  ...     0      0     0     0      0     0      0   
          1               0  ...     0      0     0     0      0     1      0   
          2               0  ...     0      0     0     0      0     0      0   
          3               0  ...     0      0     0     0      0     0      0   
          4               0  ...     0      0     0     0      0     0      0   
...                     ...  ...   ...    ...   ...   ...    ...   ...    ...   
TOM THUMB 17              0  ...     1      0     0     0      0     0      0   
          18              0  ...     1      0     0     0      0     0      0   
          19              0  ...     4      0     0     0      0     0      1   
          20              0  ...     0      0     0     0      0     0      1   
          21              0  ...     0      0     0     0      0     0      0   

                    year  years  youth  
doc_title para_num                      
ASHPUTTEL 0            0      0      0  
          1            0      0      0  
          2            0      0      0  
          3            0      0      0  
          4            0      0      0  
...                  ...    ...    ...  
TOM THUMB 17           0      0      0  
          18           0      0      0  
          19           0      0      0  
          20           0      0      0  
          21           0      0      0  

[889 rows x 280 columns]

## Generate Model with 20 Topics

In [178]:
n_topics = 5
max_iter = 100
n_top_terms = 5
TNAMES = [f"T{str(x).zfill(len(str(n_topics)))}" for x in range(n_topics)]

In [179]:
if model_type == 'lda':
    topic_engine = LDA(n_components=n_topics, max_iter=max_iter)
elif model_type == 'nmf':
    topic_engine = NMF(n_components=n_topics, max_iter=max_iter)
topic_model = topic_engine.fit_transform(count_model)

## THETA

In [180]:
THETA = pd.DataFrame(topic_model, index=DOCS.index, columns=TNAMES)
THETA.columns.name = 'topic_id'
THETA.sample(10).T.style.background_gradient(cmap=colors, axis=None)

doc_title,THE TWELVE DANCING PRINCESSES,MOTHER HOLLE,FREDERICK AND CATHERINE,ASHPUTTEL,CAT-SKIN,THE JUNIPER-TREE,HANS IN LUCK,THE WHITE SNAKE,THE SEVEN RAVENS,THE RAVEN
para_num,6,52,0,3,6,87,15,11,4,3
topic_id,,,,,,,,,,
T0,0.007797,0.066960,0.279254,0.012634,0.006786,0.100000,0.066667,0.022715,0.012655,0.837156
T1,0.452383,0.066792,0.015489,0.949589,0.972828,0.100985,0.066963,0.909803,0.012784,0.040264
T2,0.373761,0.066667,0.015458,0.012525,0.006750,0.101397,0.733036,0.022374,0.403061,0.040518
T3,0.158308,0.732914,0.015472,0.012552,0.006807,0.100000,0.066667,0.022561,0.012967,0.041372
T4,0.007751,0.066667,0.674327,0.012700,0.006829,0.597618,0.066667,0.022547,0.558534,0.040690


## PHI

In [181]:
PHI = pd.DataFrame(topic_engine.components_, columns=TERMS, index=TNAMES)
PHI.index.name = 'topic_id'
PHI.columns.name = 'term_str'
PHI.T.sample(10).T.style.background_gradient(cmap=colors, axis=None)

term_str,peasant,sword,help,joy,neck,eyes,moon,cup,tailor,tail
topic_id,,,,,,,,,,
T0,45.197252,0.200341,1.312977,0.200906,1.403380,11.169471,0.203104,3.619078,0.200329,0.200984
T1,0.200001,9.048867,5.363887,10.680907,3.915814,20.692197,7.346539,8.057964,0.201471,5.406845
T2,0.200953,0.240948,0.214928,16.715004,7.890144,0.205972,0.201732,0.202033,67.196821,0.203454
T3,0.201148,3.307564,4.907782,0.202322,1.746972,21.582760,13.047328,1.911967,0.201281,12.986326
T4,0.200647,0.202280,0.200427,0.200860,7.043689,8.349600,0.201298,0.208958,0.200098,0.202392


## Get Top Terms By Topic

In [182]:
TOPICS = PHI.stack().groupby('topic_id')\
    .apply(lambda x: ' '.join(x.sort_values(ascending=False).head(n_top_terms).reset_index().term_str))\
    .to_frame('top_terms')
TOPICS

,top_terms
topic_id,
T0,wife man head mother peasant
T1,king father son princess daughter
T2,tree tailor man dwarf way
T3,door children house wolf woman
T4,castle day man bird cat
